# AnionXAS FEFF v2 — local curation and scratch-encoder training

This notebook runs the versioned v2 workflow on your laptop:

1. **Curation** (`extract_figshare_feff_v2.py`): rebuild the eight-element FEFF target package directly from the raw Figshare `xas.json.tgz`. Duplicate keys, conflicting structures, symmetry-equivalent sites, pointwise 2.5-sigma outliers, and negative targets are all handled with explicit rejections.
2. **Preflight** for the scratch-encoder training pipeline (`train_m3gnet_xas_pipeline_v2.py`).
3. **Training** (you launch it): the showcase recipe with a **randomly initialized encoder — no pretrained M3GNet weights are ever loaded**.
4. **Evaluation**: UniversalXAS and Tuned-UniversalXAS heads, eta against the train-mean baseline.

Targets: 141 points, 0–35 eV, 0.25 eV spacing, absolute per-element starts (paper Table S1 plus calibrated offset). Test data is used exactly once, after validation-based selection.


## 0. Configuration

Set `ARCHIVE` to your local copy of the Figshare `xas.json.tgz` and `PACKAGE_DIR` to the package output location. Curation reads the archive; it never modifies it.

In [ ]:
from pathlib import Path
import json

ARCHIVE = Path(r"C:/path/to/xas.json.tgz").resolve()
PACKAGE_DIR = Path(r"C:/path/to/anionxas_feff_v2_package").resolve()
RUN_ROOT = Path(r"C:/path/to/output/training").resolve()
SEED = 42
print("archive:", ARCHIVE, ARCHIVE.is_file())
print("package dir:", PACKAGE_DIR)
print("run root:", RUN_ROOT)

## 1. Curation (long: streams the whole archive twice)

The curator writes only into `PACKAGE_DIR`; the archive stays read-only. Symmetry collapse needs `pymatgen` (`pip install pymatgen`). For a quick smoke run, add `"--limit", "20000"` to the command.

In [ ]:
RUN_CURATION = False  # set True to build the package

import subprocess, sys
ROOT = Path.cwd().parent if (Path.cwd().name == "tutorial_omnixas") else Path.cwd()

if RUN_CURATION:
    cmd = [sys.executable, str(ROOT / "tutorial_omnixas" / "extract_figshare_feff_v2.py"),
           "--archive", str(ARCHIVE),
           "--output-dir", str(PACKAGE_DIR),
           "--seed", str(SEED)]
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True)
else:
    print("curation skipped (RUN_CURATION = False)")

## 2. Package inspection

In [ ]:
import numpy as np

package = np.load(PACKAGE_DIR / "targets_141.npz", allow_pickle=True)
elements = package["elements"]
splits = np.array(["train", "val", "test"])[package["split_codes"]]
for element in sorted(set(elements)):
    counts = {s: int((splits[elements == element] == s).sum()) for s in ("train", "val", "test")}
    print(f"{element:>3}: {counts}")
print("total rows:", len(elements))
report = json.loads((PACKAGE_DIR / "report.json").read_text())
print("format:", report["format_version"], "| rejections:", report["counts"]["rejections"])

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for element in sorted(set(elements)):
    mask = elements == element
    axes[0].plot(package["energies"], package["spectras"][mask].mean(0), label=element)
axes[0].set_title("mean spectrum per element")
axes[0].set_xlabel("E relative (eV)")
axes[0].legend(fontsize=7)
order = sorted(set(elements))
axes[1].bar(order, [int((elements == e).sum()) for e in order])
axes[1].set_title("rows per element")
plt.tight_layout()
plt.show()

## 3. Preflight (no training)

In [ ]:
import subprocess, sys
ROOT = Path.cwd().parent if (Path.cwd().name == "tutorial_omnixas") else Path.cwd()
cmd = [sys.executable, str(ROOT / "tutorial_omnixas" / "train_m3gnet_xas_pipeline_v2.py"),
       "--data-root", str(PACKAGE_DIR),
       "--output-root", str(RUN_ROOT),
       "--preflight"]
print("$", " ".join(cmd))
subprocess.run(cmd, check=True)

## 4. Training (scratch encoder — launch yourself)

The encoder starts from random initialization; no pretrained weights are loaded. Defaults follow the showcase recipe: AdamW 1e-3, weight decay 1e-5, ReduceLROnPlateau, balanced element batches, baseline-normalized loss plus a derivative term. At default epochs this is a long run on one GPU. Set `RUN_TRAINING = True` when ready.

In [ ]:
RUN_TRAINING = False  # set True to start the full run
GPU = "0"             # CUDA device index, or None for automatic
NUM_WORKERS = 8
EXTRA_ARGS: list[str] = []  # e.g. ["--precompute-graphs"]

import subprocess, sys
ROOT = Path.cwd().parent if (Path.cwd().name == "tutorial_omnixas") else Path.cwd()

if RUN_TRAINING:
    cmd = [sys.executable, str(ROOT / "tutorial_omnixas" / "train_m3gnet_xas_pipeline_v2.py"),
           "--data-root", str(PACKAGE_DIR),
           "--output-root", str(RUN_ROOT),
           "--seed", str(SEED),
           "--num-workers", str(NUM_WORKERS)]
    if GPU is not None:
        cmd += ["--gpu", str(GPU)]
    cmd += EXTRA_ARGS
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True)
else:
    print("training skipped (RUN_TRAINING = False)")

## 5. Evaluation and metrics

Evaluates the completed run's checkpoints once on val and test.

In [ ]:
RUN_NAME = f"m3gnet_xas_v2_seed{SEED}"

import subprocess, sys
ROOT = Path.cwd().parent if (Path.cwd().name == "tutorial_omnixas") else Path.cwd()
cmd = [sys.executable, str(ROOT / "tutorial_omnixas" / "train_m3gnet_xas_pipeline_v2.py"),
       "--data-root", str(PACKAGE_DIR),
       "--output-root", str(RUN_ROOT),
       "--run-name", RUN_NAME,
       "--evaluate"]
print("$", " ".join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
import pandas as pd

run_dir = RUN_ROOT / RUN_NAME
for name in ("universal_validation", "universal_test", "tuned_validation", "tuned_test"):
    csv_path = run_dir / f"{name}.csv"
    if csv_path.is_file():
        frame = pd.read_csv(csv_path)
        eta_columns = [c for c in frame.columns if c.endswith("_eta")]
        print(f"== {name} ==")
        print(frame[["dataset", *eta_columns]].to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

fig, ax = plt.subplots(figsize=(8, 3.5))
val = pd.read_csv(run_dir / "universal_validation.csv")
test = pd.read_csv(run_dir / "universal_test.csv")
width = 0.4
x = np.arange(len(val))
ax.bar(x - width / 2, val["val_eta"], width, label="val eta")
ax.bar(x + width / 2, test["test_eta"], width, label="test eta")
ax.set_xticks(x, [d.replace("_FEFF", "") for d in val["dataset"]])
ax.axhline(1.0, color="gray", linestyle=":", label="baseline (eta=1)")
ax.set_ylabel("eta")
ax.set_title("UniversalXAS (v2 scratch encoder)")
ax.legend()
plt.tight_layout()
plt.show()

## Notes

- The 141-point package is the canonical v2 target. Do not mix it with 200-point runs.
- Provenance for every run lands in `RUN_ROOT/RUN_NAME/provenance.json` (seed, package report, encoder-init policy, arguments).
- Test metrics are computed once by the `--evaluate` stage; never select checkpoints on them.